# 경사하강법 · 오차역전파 · 최적화 — 기울기로 가중치를 고치는 방법

수업 메모 `경사하강법` · `다중_분류_손실` 과 교재 내용을 실행 가능하도록 정리 · 개선한 노트입니다.

- 📖 교재 : `딥러닝` 교재8 **p.46~48** (배치 vs 미니배치) · **p.54~78** (편미분 · 기울기 · 경사하강법 · 행렬 모양 · 계산 그래프 · 연쇄법칙 · 오차역전파 · SGD · Momentum · AdaGrad · Adam)
- 🔗 이어지는 노트 : [`mlp_titanic_binary.ipynb`](./mlp_titanic_binary.ipynb) (PyTorch MLP 이진 분류) · [`mlp_diabetes_iris.ipynb`](./mlp_diabetes_iris.ipynb) (이진 · 다중 분류 · 회귀)
- 📝 메모 · 교재에서 틀렸거나 헷갈리는 부분은 그 자리에 **📝 고친 부분** 으로 적고, 맨 아래에 표로 모았습니다.
- 🎬 애니메이션은 [`images/`](./images/) 폴더 (이 노트와 같은 함수 · 같은 숫자로 만든 것)

In [1]:
# 한글 폰트 설정 (Windows: Malgun Gothic / macOS: AppleGothic)
import platform
import matplotlib.pyplot as plt
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지

In [2]:
import numpy as np
import torch
np.set_printoptions(precision=4, suppress=True)

## 1. 배치 vs 미니배치 (교재 p.46~48)

손실을 **몇 개의 데이터로** 계산한 뒤 가중치를 한 번 고칠까?

| 방식 | 한 번 업데이트에 쓰는 데이터 | 1 에포크(전체 1번) 동안 업데이트 횟수 |
|---|---|---|
| 배치 학습 | 전체 N 개 | **1 번** |
| 미니배치 학습 | 무작위 n 개 (배치 크기) | **⌈N / n⌉ 번** (마지막 묶음은 남은 개수만큼) |

타이타닉 실습의 학습용 데이터 N = 1046, 배치 크기 n = 32 로 계산해 봅니다.

In [3]:
N, n = 1046, 32                      # 타이타닉 학습용 데이터 수, 미니배치 크기
rng = np.random.default_rng(0)       # 시드 고정
idx = rng.permutation(N)             # 매 에포크 순서를 섞는다 (DataLoader 의 shuffle=True)

batches = [idx[i:i + n] for i in range(0, N, n)]   # n 개씩 잘라 묶음 만들기
print(f'1 에포크 업데이트 횟수: {len(batches)} 번  (N / n = {N / n:.2f} → 올림)')
print(f'묶음 크기: 처음 {len(batches[0])} 개 … 마지막 {len(batches[-1])} 개')
print(f'배치 학습이면 1 에포크에 업데이트 1 번')

1 에포크 업데이트 횟수: 33 번  (N / n = 32.69 → 올림)
묶음 크기: 처음 32 개 … 마지막 22 개
배치 학습이면 1 에포크에 업데이트 1 번


> **📝 고친 부분** — 교재 p.47 과 p.48 은 같은 슬라이드가 두 번 들어가 있다. 또 "N/n 번" 은 나누어떨어질 때 이야기이고, 실제로는 **올림** (1046 / 32 = 32.7 → 33 번, 마지막 묶음 22 개). PyTorch `DataLoader` 도 같은 방식(`drop_last=False` 기본값).

## 2. 편미분과 기울기 (교재 p.54~56)

변수가 여러 개면 **하나만 움직이고 나머지는 상수로 고정** 해서 미분한다 → **편미분** $\frac{\partial f}{\partial x_0}$.
모든 변수의 편미분을 한 배열로 묶은 것이 **기울기(gradient)**.

교재 예시 $f(x_0, x_1) = x_0^2 + x_1^2$, 점 $(3, 4)$ → $\left(\frac{\partial f}{\partial x_0}, \frac{\partial f}{\partial x_1}\right) = (2 \cdot 3,\ 2 \cdot 4) = (6, 8)$

In [4]:
def f2(x):
    """교재 p.55 의 가상 손실 함수 f(x0, x1) = x0² + x1²"""
    return np.sum(x ** 2)

def numerical_gradient(f, x, h=1e-4):
    """변수마다 하나씩 움직여 중앙 차분 → 편미분을 모은 기울기 배열"""
    x = x.astype(float).copy()
    grad = np.zeros_like(x)
    for i in range(x.size):
        orig = x.flat[i]
        x.flat[i] = orig + h; fxh1 = f(x)      # i 번째 변수만 +h
        x.flat[i] = orig - h; fxh2 = f(x)      # i 번째 변수만 -h
        grad.flat[i] = (fxh1 - fxh2) / (2 * h) # 나머지 변수는 그대로(상수 취급)
        x.flat[i] = orig                       # 원래 값으로 되돌리기
    return grad

print('(3, 4) 의 기울기:', numerical_gradient(f2, np.array([3.0, 4.0])))   # 해석적: (6, 8)
print('(0, 2) 의 기울기:', numerical_gradient(f2, np.array([0.0, 2.0])))
print('(3, 0) 의 기울기:', numerical_gradient(f2, np.array([3.0, 0.0])))

(3, 4) 의 기울기: [6. 8.]
(0, 2) 의 기울기: [0. 4.]
(3, 0) 의 기울기: [6. 0.]


기울기 벡터는 **함수 값이 가장 크게 늘어나는 방향** 을 가리킨다. 그래서 손실을 줄이려면 **기울기의 반대 방향(−)** 으로 움직인다.

## 3. 경사하강법 — 메모의 포물선 (교재 p.57~58)

$$W_{new} = W_{old} - \eta \frac{\partial L}{\partial W} \qquad (\eta = \text{학습률, 한 걸음의 보폭 비율})$$

메모 `경사하강법` : 회귀 출력 뉴런의 예측 $y = wx$ ($x = 1$), 정답 1 → 손실 $L = (1 - w)^2$ (포물선).
**w = 3 에서 출발** (손실 4) → w = 2 (손실 1) → **w = 1 (손실 0, 학습 목표)**.
기울기 $\frac{dL}{dw} = -2(1 - w)$ 를 이용해 학습률 0.1 로 한 걸음씩 내려가 봅니다.

In [5]:
def loss_memo(w, x=1.0, t=1.0):
    """메모의 손실 L = (정답 - y)², y = wx"""
    return (t - w * x) ** 2

def grad_memo(w, x=1.0, t=1.0):
    """dL/dw = -2x(t - wx)  (연쇄법칙: 바깥 제곱 미분 × 안쪽 wx 미분)"""
    return -2 * x * (t - w * x)

def gradient_descent(w0, lr, steps):
    """경사하강법: w ← w - lr × 기울기, 지나간 w 를 모두 기록"""
    ws = [w0]
    for _ in range(steps):
        ws.append(ws[-1] - lr * grad_memo(ws[-1]))
    return np.array(ws)

ws = gradient_descent(w0=3.0, lr=0.1, steps=20)
for step in [0, 1, 2, 3, 5, 10, 20]:
    w = ws[step]
    print(f'step {step:2d} : w = {w:.4f}   손실 = {loss_memo(w):.4f}   기울기 = {grad_memo(w):+.4f}   보폭 = {0.1 * abs(grad_memo(w)):.4f}')

step  0 : w = 3.0000   손실 = 4.0000   기울기 = +4.0000   보폭 = 0.4000
step  1 : w = 2.6000   손실 = 2.5600   기울기 = +3.2000   보폭 = 0.3200
step  2 : w = 2.2800   손실 = 1.6384   기울기 = +2.5600   보폭 = 0.2560
step  3 : w = 2.0240   손실 = 1.0486   기울기 = +2.0480   보폭 = 0.2048
step  5 : w = 1.6554   손실 = 0.4295   기울기 = +1.3107   보폭 = 0.1311
step 10 : w = 1.2147   손실 = 0.0461   기울기 = +0.4295   보폭 = 0.0429
step 20 : w = 1.0231   손실 = 0.0005   기울기 = +0.0461   보폭 = 0.0046


![경사하강법 애니메이션](./images/anim_01_gradient_descent.gif)

결과 해석
- 처음엔 기울기 +4 → 보폭 0.4 로 크게, 바닥에 가까워질수록 기울기 · 보폭이 **같이 작아진다** (학습률은 0.1 그대로).
- 메모 결론 "손실이 커질수록 기울기도 커진다" 는 이 포물선에서 맞다 : $|\text{기울기}| = 2|1 - w| = 2\sqrt{L}$.

> **📝 고친 부분** — 메모 1행 "입력 함수의 총합" → **입력 신호의 총합** ($Wx$). 또 "손실이 커질수록 기울기가 커진다" 는 **볼록한 포물선(제곱 오차)** 에서만 성립한다. 뒤의 7절처럼 언덕이 있는 손실 곡선은 손실이 큰데 기울기가 0 인 곳(언덕 꼭대기)도 있다.

### 3-1. 학습률의 크기

학습률이 너무 작으면 느리고, 너무 크면 바닥을 **넘어 반대편으로 튀고**, 1 보다 크면(이 포물선에서) **발산** 한다.

In [6]:
for lr in [0.01, 0.1, 0.5, 0.9, 1.1]:
    ws_lr = gradient_descent(3.0, lr, 20)
    print(f'lr = {lr:<4} → 20 걸음 뒤 w = {ws_lr[-1]:>10.4f}   손실 = {loss_memo(ws_lr[-1]):.4g}')

lr = 0.01 → 20 걸음 뒤 w =     2.3352   손실 = 1.783
lr = 0.1  → 20 걸음 뒤 w =     1.0231   손실 = 0.0005317
lr = 0.5  → 20 걸음 뒤 w =     1.0000   손실 = 0
lr = 0.9  → 20 걸음 뒤 w =     1.0231   손실 = 0.0005317
lr = 1.1  → 20 걸음 뒤 w =    77.6752   손실 = 5879


![학습률 비교 애니메이션](./images/anim_02_learning_rate.gif)

- lr = 0.5 는 이 포물선에서 **한 번에** 바닥(w = 1)에 도착하는 특별한 값 ($w - 0.5 \cdot 2(w-1) = 1$).
- lr = 0.9 는 바닥을 넘나들며(3 → −0.6 → 2.28 …) 지그재그로 수렴, lr = 1.1 은 점점 멀어져 **발산**.

### 3-2. 변수 2 개 — 기울기 반대 방향으로 내려가기

In [7]:
def gradient_descent_nd(f, init_x, lr=0.1, steps=100):
    """기울기 배열을 쓰는 경사하강법 (교재 p.58 공식을 변수 여러 개로)"""
    x = init_x.astype(float).copy()
    for _ in range(steps):
        x -= lr * numerical_gradient(f, x)
    return x

print('lr 0.1 , 100 걸음:', gradient_descent_nd(f2, np.array([-3.0, 4.0]), lr=0.1))    # (0, 0) 근처
print('lr 10  , 100 걸음:', gradient_descent_nd(f2, np.array([-3.0, 4.0]), lr=10.0))   # 발산
print('lr 1e-10, 100 걸음:', gradient_descent_nd(f2, np.array([-3.0, 4.0]), lr=1e-10)) # 거의 안 움직임

lr 0.1 , 100 걸음: [-0.  0.]
lr 10  , 100 걸음: [-2.5898e+13 -1.2952e+12]
lr 1e-10, 100 걸음: [-3.  4.]


## 4. 다층 신경망의 행렬 모양 (교재 p.59~61)

| 이름 | 모양 | 뜻 |
|---|---|---|
| X | (4, 2) | 미니배치 4 개 × 입력 특성 2 개 |
| W1 | (2, 3) | 입력 2 → 은닉층 노드 3 |
| Z = ReLU(X·W1 + b1) | (4, 3) | |
| W2 | (3, 3) | 은닉 3 → 출력 3 (클래스 3 개) |
| Y = softmax(Z·W2 + b2) | (4, 3) | 데이터마다 클래스 3 개의 확률 |
| T | (4, 3) | 원-핫 정답 |
| **∂L/∂W1 · ∂L/∂W2** | **(2, 3) · (3, 3)** | 기울기는 **가중치와 모양이 같다** |

수치 미분으로 기울기를 직접 구해 모양을 확인합니다.

In [8]:
def softmax(a):
    a = a - a.max(axis=1, keepdims=True)    # 오버플로 방지 (값은 그대로)
    e = np.exp(a)
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy(y, t):
    return -np.sum(t * np.log(y + 1e-7)) / y.shape[0]   # 미니배치 평균

rng = np.random.default_rng(0)
X = rng.normal(size=(4, 2))                       # 미니배치 4 개, 특성 2 개
T = np.eye(3)[[0, 1, 2, 1]]                       # 원-핫 정답 (4, 3)
params = {'W1': rng.normal(scale=0.5, size=(2, 3)), 'b1': np.zeros(3),
          'W2': rng.normal(scale=0.5, size=(3, 3)), 'b2': np.zeros(3)}

def loss_fn(_):
    Z = np.maximum(0, X @ params['W1'] + params['b1'])   # 은닉층 + ReLU → (4, 3)
    Y = softmax(Z @ params['W2'] + params['b2'])        # 출력층 + 소프트맥스 → (4, 3)
    return cross_entropy(Y, T)

print('손실 L =', round(loss_fn(None), 4), '(숫자 하나, 스칼라)')
for name in ['W1', 'b1', 'W2', 'b2']:
    # params 의 원소를 하나씩 직접 ±h 움직이며 손실 변화를 잰다 (2절 numerical_gradient 와 같은 방식)
    grad = np.zeros_like(params[name])
    for i in range(grad.size):
        orig = params[name].flat[i]
        params[name].flat[i] = orig + 1e-4; l1 = loss_fn(None)
        params[name].flat[i] = orig - 1e-4; l2 = loss_fn(None)
        params[name].flat[i] = orig
        grad.flat[i] = (l1 - l2) / 2e-4
    print(f'{name}: 가중치 모양 {params[name].shape} → 기울기 모양 {grad.shape}')

손실 L = 1.112 (숫자 하나, 스칼라)
W1: 가중치 모양 (2, 3) → 기울기 모양 (2, 3)
b1: 가중치 모양 (3,) → 기울기 모양 (3,)
W2: 가중치 모양 (3, 3) → 기울기 모양 (3, 3)
b2: 가중치 모양 (3,) → 기울기 모양 (3,)


## 5. 다중 분류 손실 — 메모 `다중_분류_손실`

정답이 원-핫이면 **정답 클래스의 예측 확률 하나만** 손실에 들어간다.

In [9]:
def cee(y, t):
    return -np.sum(t * np.log(y))

# 메모 1 : 개 · 고양이 · 호랑이, 정답 고양이
y3, t3 = np.array([0.1, 0.7, 0.2]), np.array([0, 1, 0])
print('항별 t·log(y):', t3 * np.log(y3), '→ 손실', round(cee(y3, t3), 4))

# 메모 2 : 고양이 · 강아지 · 사람 · 자동차 · 집, 정답 사람, p3 = 0.7
y5, t5 = np.array([0.05, 0.1, 0.7, 0.1, 0.05]), np.array([0, 0, 1, 0, 0])
print('Σ t·log(p) =', round(np.sum(t5 * np.log(y5)), 4), '(음수)  →  손실 = -Σ =', round(cee(y5, t5), 4), '(양수)')

항별 t·log(y): [-0.     -0.3567 -0.    ] → 손실 0.3567
Σ t·log(p) = -0.3567 (음수)  →  손실 = -Σ = 0.3567 (양수)


> **📝 고친 부분** — 메모 마지막 줄 `t·log(P) = … = 1·log(p3) = -log(0.7) (양수)` 는 **부호가 섞였다**. $\sum t \log P = \log 0.7 = -0.357$ (음수)이고, 교차 엔트로피는 앞에 **−** 를 붙여 $E = -\log 0.7 = 0.357$ (양수). 또 메모 그림의 $y_1 = W_{11}x_1 + W_{12}x_2 + b_1 = n_1 \to 0.1$ 에서 $n_1$ 은 **점수(입력 신호의 총합)** 이고, 0.1 이라는 **확률은 소프트맥스를 거친 뒤** 의 값이다. 메모 번호 `1)` 이 두 번 → 1), 2).

## 6. 계산 그래프 · 연쇄법칙 · 역전파 (교재 p.62~67)

사과 1개 100원 × 2개 × 소비세 1.1 = **220원**. 각 노드는 **자기 입력과 출력만** 신경 쓴다(국소적 계산).
곱셈 노드의 역전파 : 위에서 온 미분 × **상대편 입력값** (서로 바꿔서 곱).

In [10]:
class MulLayer:
    """곱셈 노드: 순전파 x*y, 역전파 dout*y, dout*x (입력을 서로 바꿔 곱한다)"""
    def forward(self, x, y):
        self.x, self.y = x, y          # 역전파에 쓰려고 입력을 기억
        return x * y
    def backward(self, dout):
        return dout * self.y, dout * self.x

class AddLayer:
    """덧셈 노드: 위에서 온 미분을 그대로 양쪽에 흘려보낸다"""
    def forward(self, x, y):
        return x + y
    def backward(self, dout):
        return dout, dout

apple, apple_num, tax = 100, 2, 1.1
mul_apple, mul_tax = MulLayer(), MulLayer()

# 순전파 (왼쪽 → 오른쪽)
apple_price = mul_apple.forward(apple, apple_num)   # 200
price = mul_tax.forward(apple_price, tax)           # 220
print('순전파 결제 금액:', round(price, 2))

# 역전파 (오른쪽 → 왼쪽), 맨 끝 d(price)/d(price) = 1 에서 출발
dprice = 1
dapple_price, dtax = mul_tax.backward(dprice)       # 1.1, 200
dapple, dapple_num = mul_apple.backward(dapple_price)  # 2.2, 110
print(f'∂금액/∂사과값 = {dapple:.1f}   ∂금액/∂개수 = {dapple_num:.0f}   ∂금액/∂소비세 = {dtax:.0f}')

순전파 결제 금액: 220.0
∂금액/∂사과값 = 2.2   ∂금액/∂개수 = 110   ∂금액/∂소비세 = 200


해석 : 사과값이 1원 오르면 금액은 **2.2원** (= 2개 × 1.1) 오른다. 연쇄법칙 $\frac{\partial y}{\partial x} = \frac{\partial y}{\partial u} \times \frac{\partial u}{\partial x} = 1.1 \times 2$.

PyTorch 의 `loss.backward()` 가 바로 이 일을 자동으로 한다(자동 미분). 같은 값이 나오는지 확인 :

In [11]:
a = torch.tensor(100.0, requires_grad=True)   # 미분을 추적할 변수
n_ = torch.tensor(2.0, requires_grad=True)
tx = torch.tensor(1.1, requires_grad=True)
total = a * n_ * tx                           # 순전파
total.backward()                              # 역전파
print(f'PyTorch 자동 미분 → ∂사과값 {a.grad:.1f} · ∂개수 {n_.grad:.0f} · ∂소비세 {tx.grad:.0f}')

PyTorch 자동 미분 → ∂사과값 2.2 · ∂개수 110 · ∂소비세 200


## 7. 퍼셉트론 하나의 가중치 기울기 (교재 p.68~70)

$$\frac{\partial L}{\partial w} = \underbrace{\frac{\partial L}{\partial a}}_{\text{뒤에서 온 오차}} \times \underbrace{\frac{\partial a}{\partial z}}_{\text{활성화 함수 미분}} \times \underbrace{\frac{\partial z}{\partial w}}_{= x}$$

퍼셉트론 1개 : $z = wx + b$, $a = \sigma(z)$, 정답 $t = 1$, 손실 $L = -\log a$ (BCE).
세 가지 방법(역전파 손계산 · 수치 미분 · PyTorch)으로 같은 값이 나오는지 비교합니다.

In [12]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))
x, w, b, t = 2.0, 0.5, -0.3, 1.0

# 순전파
z = w * x + b
a = sigmoid(z)
L = -(t * np.log(a) + (1 - t) * np.log(1 - a))
print(f'z = {z:.3f}  a = σ(z) = {a:.4f}  L = {L:.4f}')

# ① 역전파 (국소적 미분 3개의 곱)
dL_da = -(t / a) + (1 - t) / (1 - a)      # BCE 미분
da_dz = a * (1 - a)                       # 시그모이드 미분
dz_dw = x                                 # z = wx + b 를 w 로 미분 = 입력값 x
grad_bp = dL_da * da_dz * dz_dw
print(f'① 역전파   : {dL_da:.4f} × {da_dz:.4f} × {dz_dw:.1f} = {grad_bp:.6f}   (= (a - t)·x = {(a - t) * x:.6f})')

# ② 수치 미분
Lw = lambda w_: -np.log(sigmoid(w_ * x + b))
print(f'② 수치 미분 : {(Lw(w + 1e-4) - Lw(w - 1e-4)) / 2e-4:.6f}')

# ③ PyTorch 자동 미분
wt = torch.tensor(w, requires_grad=True)
Lt = torch.nn.functional.binary_cross_entropy(torch.sigmoid(wt * x + b), torch.tensor(t))
Lt.backward()
print(f'③ PyTorch   : {wt.grad.item():.6f}')

z = 0.700  a = σ(z) = 0.6682  L = 0.4032
① 역전파   : -1.4966 × 0.2217 × 2.0 = -0.663624   (= (a - t)·x = -0.663624)
② 수치 미분 : -0.663624
③ PyTorch   : -0.663625


세 값이 같다. 시그모이드 + BCE 를 붙이면 앞 두 항이 약분되어 **(a − t)·x** 로 아주 간단해진다 — 그래서 PyTorch 도 둘을 합친 `BCEWithLogitsLoss` 를 쓴다.

> **📝 고친 부분** — 교재 p.69 번호가 `1) ② ③` 으로 섞여 있다 → ① ② ③. 교재 p.65 "u 가 분수처럼 **약분**" 은 외우기 쉬운 비유일 뿐, 미분 기호는 실제 분수가 아니다(결과가 같게 나오도록 정의된 것).

## 8. 최적화 알고리즘 — SGD · Momentum · AdaGrad · Adam (교재 p.71~78)

| 이름 | 업데이트 | 아이디어 |
|---|---|---|
| SGD | $W \leftarrow W - \eta g$ | 기울기 반대로 정직하게 한 걸음 |
| Momentum | $v \leftarrow \alpha v - \eta g,\ W \leftarrow W + v$ | 굴러온 **속도(관성)** 를 기억 ($\alpha = 0.9$) |
| AdaGrad | $h \leftarrow h + g^2,\ W \leftarrow W - \frac{\eta}{\sqrt{h} + \epsilon} g$ | 많이 움직인 가중치는 보폭을 줄인다 |
| Adam | $m$ (기울기 평균) + $v$ (기울기² 평균, 오래된 값은 잊음) | Momentum + 보폭 자동 조절 |

언덕이 하나 있는 손실 곡선 $L(w) = \frac{(w^2 - 4)^2}{4} + w$ 로 실험한다.
- 가짜 골짜기(로컬 미니멈) **w ≈ 1.86 (손실 1.93)**, 진짜 골짜기(전역 최소) **w ≈ −2.11 (손실 −2.06)**, 언덕 꼭대기 w ≈ 0.25
- 모두 **w = 3.5** 에서 출발, 100 걸음

In [13]:
L_dw = lambda w: (w ** 2 - 4) ** 2 / 4 + w      # 언덕이 있는 손실 곡선
g_dw = lambda w: w * (w ** 2 - 4) + 1           # 기울기

def optimize(kind, lr, w=3.5, steps=100):
    """4 가지 최적화 알고리즘을 같은 출발점에서 실행, 지나간 w 기록"""
    v = h = m = s = 0.0
    path = [w]
    for t_ in range(1, steps + 1):
        g = g_dw(w)
        if kind == 'SGD':
            w -= lr * g
        elif kind == 'Momentum':
            v = 0.9 * v - lr * g                 # 속도 = 관성 × 이전 속도 - 학습률 × 기울기
            w += v
        elif kind == 'AdaGrad':
            h += g * g                           # 기울기 제곱 누적 (주행거리 계기판)
            w -= lr * g / (np.sqrt(h) + 1e-7)
        elif kind == 'Adam':
            m = 0.9 * m + 0.1 * g                # 기울기의 이동 평균 (관성)
            s = 0.999 * s + 0.001 * g * g        # 기울기 제곱의 이동 평균 (오래된 값은 잊음)
            m_hat, s_hat = m / (1 - 0.9 ** t_), s / (1 - 0.999 ** t_)   # 초기 편향 보정
            w -= lr * m_hat / (np.sqrt(s_hat) + 1e-8)
        if abs(w) > 1e3:                         # 발산하면 멈춤 (overflow 방지)
            path.append(np.inf)
            break
        path.append(w)
    return np.array(path)

settings = {'SGD': 0.05, 'Momentum': 0.02, 'AdaGrad': 0.5, 'Adam': 0.3}   # 애니메이션과 같은 값
for kind, lr in settings.items():
    p = optimize(kind, lr)
    where = '진짜 골짜기 ✅' if p[-1] < 0 else '가짜 골짜기 ❌'
    print(f'{kind:9s} lr={lr:<5} → 최종 w = {p[-1]:+.3f}  손실 = {L_dw(p[-1]):+.3f}  {where}  (가장 왼쪽까지 간 w = {p.min():+.2f})')

SGD       lr=0.05  → 최종 w = +1.861  손실 = +1.933  가짜 골짜기 ❌  (가장 왼쪽까지 간 w = +1.86)
Momentum  lr=0.02  → 최종 w = -2.122  손실 = -2.059  진짜 골짜기 ✅  (가장 왼쪽까지 간 w = -3.00)
AdaGrad   lr=0.5   → 최종 w = +1.861  손실 = +1.933  가짜 골짜기 ❌  (가장 왼쪽까지 간 w = +1.86)
Adam      lr=0.3   → 최종 w = +1.854  손실 = +1.933  가짜 골짜기 ❌  (가장 왼쪽까지 간 w = +0.91)


![최적화 알고리즘 비교 애니메이션](./images/anim_03_optimizers.gif)

결과 해석
- **SGD** 는 가짜 골짜기 바닥에서 기울기가 0 이 되어 멈춘다 (교재 p.73 그대로).
- **Momentum** (lr 0.02) 은 쌓인 속도로 언덕을 넘어 진짜 골짜기로 간다.
- **AdaGrad** 는 보폭이 계속 줄어 가짜 골짜기에 머문다 (교재 p.77 의 한계).
- **Adam** (lr 0.3) 은 이 설정에서는 언덕을 **넘지 못했다**.

그런데 학습률을 바꾸면 결과가 달라진다 — 언덕을 넘는지는 **알고리즘 + 학습률 + 출발점** 에 달렸다.

In [14]:
print(f'{"lr":>6} | ' + ' | '.join(f'{k:^9}' for k in settings))
for lr in [0.01, 0.02, 0.05, 0.1, 0.3, 0.5, 1.0]:
    row = []
    for kind in settings:
        p = optimize(kind, lr)
        row.append('진짜 ✅' if p[-1] < 0 else ('발산 💥' if not np.isfinite(p[-1]) or abs(p[-1]) > 10 else '가짜 ❌'))
    print(f'{lr:>6} | ' + ' | '.join(f'{r:^8}' for r in row))

    lr |    SGD    | Momentum  |  AdaGrad  |   Adam   
  0.01 |   가짜 ❌   |   진짜 ✅   |   가짜 ❌   |   가짜 ❌  
  0.02 |   가짜 ❌   |   진짜 ✅   |   가짜 ❌   |   가짜 ❌  
  0.05 |   가짜 ❌   |   가짜 ❌   |   가짜 ❌   |   가짜 ❌  
   0.1 |   가짜 ❌   |   진짜 ✅   |   가짜 ❌   |   가짜 ❌  
   0.3 |   발산 💥   |   발산 💥   |   가짜 ❌   |   가짜 ❌  
   0.5 |   발산 💥   |   발산 💥   |   가짜 ❌   |   진짜 ✅  
   1.0 |   발산 💥   |   발산 💥   |   가짜 ❌   |   진짜 ✅  


표 읽기 : Momentum 은 lr 0.05 에서만 실패(왼쪽으로 넘어갔다가 되돌아옴)하는 들쭉날쭉한 결과, **AdaGrad 는 어떤 lr 에서도** 가짜 골짜기, Adam 은 lr 0.5 이상에서 언덕을 넘는다. SGD · Momentum 은 lr 0.3 부터 발산하는데, Adam · AdaGrad 는 기울기 크기로 나눠 주기 때문에 큰 lr 에서도 발산하지 않는다.

> **📝 고친 부분**
> - 교재 p.75 "모멘텀은 가짜 골짜기를 **탈출하여** 진짜 목적지로 전진" → **항상 그렇지는 않다.** 위 표처럼 lr 0.05 의 Momentum 은 왼쪽으로 크게 넘어갔다가(−3.39) 다시 되돌아와 가짜 골짜기에 멈췄다. 관성은 탈출할 **가능성** 을 높일 뿐이다.
> - 교재 p.72 SGD 의 '확률적(Stochastic)' 은 **무작위로 뽑은 미니배치로 기울기를 구한다** 는 뜻이다. 공식 자체는 경사하강법과 같고, 위 실험은 데이터가 없는 함수라 사실상 '경사하강법' 이다.
> - 교재 p.72 "보폭(**업데이트된 가중치의 크기**)" → 보폭은 **학습률 × 기울기** (가중치가 바뀌는 양)이지 가중치 자체의 크기가 아니다.
> - 교재 p.78 Adam 의 "기억 상실(오래된 기울기 제곱을 깎아냄)" 은 원래 **RMSProp** 의 아이디어이고, Adam 은 여기에 Momentum 과 **초기 편향 보정** ($\hat m, \hat v$) 을 더했다.
> - 실제 딥러닝의 손실 곡선은 변수가 수백만 개라, 이런 1차원 그림처럼 가짜 골짜기에 갇히는 일보다 **평평한 안장점** 에서 느려지는 문제가 더 흔하다.

## 📝 고친 부분 정리

| 위치 | 원래 | 고친 내용 |
|---|---|---|
| 교재 p.47 · 48 | 같은 슬라이드 반복, "N/n 번" | 한 장으로, 업데이트 횟수는 **올림** (1046/32 → 33 번) |
| 메모 `경사하강법` | "입력 함수의 총합" | **입력 신호의 총합** |
| 메모 `경사하강법` | 손실이 크면 기울기도 크다 | 볼록한 포물선에서만 성립 ($|g| = 2\sqrt{L}$) |
| 메모 `다중_분류_손실` | $t \cdot \log P = \log p_3 = -\log 0.7$ (양수) | $\sum t \log P = \log 0.7$ (**음수**), $E = -\log 0.7 = 0.357$ |
| 메모 `다중_분류_손실` | $n_1 \to 0.1$ | $n_1$ 은 점수, 0.1 은 **소프트맥스 뒤** 확률 |
| 교재 p.65 | u 가 분수처럼 약분 | 비유일 뿐, 미분 기호는 분수가 아님 |
| 교재 p.69 | 번호 1) ② ③ | ① ② ③ |
| 교재 p.72 | 보폭 = 업데이트된 가중치의 크기 | 보폭 = **학습률 × 기울기** |
| 교재 p.75 | 모멘텀이 가짜 골짜기를 탈출 | 학습률에 따라 **탈출 못 할 수도** 있음 (lr 0.05 실험) |
| 교재 p.78 | Adam = Momentum + AdaGrad | 정확히는 Momentum + **RMSProp** + 편향 보정 |